# Dataset preprocessing

In [ ]:
def default_params(): 
    return {
        'dataset' : {
            'path' : '/workspaces/mica/semeru-datasets/mica',
            #CodeGEN, #CodeGPT, #bigcode_starcoder2-3b, codellama_CodeLlama-7b-hf, deepseek-ai_deepseek-coder-1.3b-base, JetBrains_Mellum-4b-base, stabilityai_stable-code-3b 
            'model': 'bigcode_starcoder2-3b'
        },
        'output_dir': '/workspaces/mica/semeru-datasets/mica/causal_analysis',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
    }
params = default_params()

### Imports

In [2]:
import os 
import pandas as pd
import lizard
import re
from tree_sitter import Language, Parser
import tree_sitter_java as tsjava
import io
import sys
from pathlib import Path
from itertools import chain

In [ ]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Analysis

In [3]:
class TreeSitterManager():
    def __init__(self, lang):
        #self.language = self.get_language(lang)
        self.language = lang
        self.parser = Parser(self.language)
        #self.parser.set_language(self.language)

    def get_ast_errors_and_deep(self, code):
        node_tree = self.parser.parse(bytes(code, "utf8"))
        return self.__detect_ast_errors_and_deep(node_tree.root_node, set())

    def __detect_ast_errors_and_deep(self, node_root, identifier_set = set(), level=0, max_level=0, count = 0):
        """Traverses the tree catch errors and evaluate tree levels"""
        # if not node_root.has_error:
        #     return [], 0
        counter = node_root.child_count

        results = []
        if node_root.type == "ERROR":
            results.append(node_root.text.decode("utf-8"))
        elif node_root.type == "identifier":
            identifier_set.add(node_root.text)
        level += 1
        for n in node_root.children:
            x, identifier_set, y, max_level, count = self.__detect_ast_errors_and_deep(n, identifier_set, level, max_level)
            max_level = max(y, max_level)
            counter += count
            if len(x) > 0:
                results.extend(x)

        return results, identifier_set, max_level, level, counter

In [ ]:
def analyze_method_java(code_string):
    temp_file_dir = f"{params['cache_dir']}/mica/lizard/{params['dataset']['model']}"
    create_folder(temp_file_dir)
    temp_file_path = f"{temp_file_dir}/temp.java"
    
    # Redirect stdout to suppress Lizard's prints
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    try:
        analysis = lizard.analyze_file.analyze_source_code(temp_file_path, code_string)
    finally:
        sys.stdout = old_stdout  # Restore stdout

    # Extract function details
    functions = []
    for function in analysis.function_list:
        functions.append({
            "fun_name": function.name,
            "complexity": function.cyclomatic_complexity,
            "nloc": function.nloc,
            "token_counts": function.token_count
        })
    
    return functions[0] if functions else None


In [5]:
def append_covariates(code_df, code_column, variable_type):
    ast_error_detector = TreeSitterManager(Language(tsjava.language()))

    def compute_covariates(row):
        java_code = row[code_column]
        lizard_results = analyze_method_java(java_code)
        try:
            ast_errors, identifier_set, ast_deep, level, count = ast_error_detector.get_ast_errors_and_deep(java_code)
        except RecursionError as e:
            print(f"RecursionError in AST analysis: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values
        except Exception as e:
            print(f"Error computing AST errors: {e}")
            ast_errors, identifier_set, ast_deep, level, count = [], set(), None, None, 0  # Default values

        return pd.Series({
            variable_type+'_complexity': lizard_results.get('complexity') if lizard_results else None,
            variable_type+'_nloc': lizard_results.get('nloc') if lizard_results else None,
            variable_type+'_token_counts': lizard_results.get('token_counts') if lizard_results else None,
            variable_type+'_ast_levels': ast_deep,
            variable_type+'_n_ast_nodes': count,
            variable_type+'_n_ast_errors': len(ast_errors) if ast_errors else 0,
            variable_type+'_ast_errors': ast_errors,
            variable_type+'_n_identifiers': len(identifier_set) if identifier_set else 0,
            variable_type+'_identifiers': identifier_set
        })
    covarites_df = code_df.apply(lambda row: compute_covariates(row), axis=1)
    code_df = pd.concat([code_df, covarites_df], axis=1)
    return code_df

In [ ]:
def format_java_code(java_code: str) -> str:
    # Remove <s> and </s> tags
    java_code = re.sub(r"</?s>", "", java_code)
    
    # Fix spaces around dots in package and import statements
    java_code = re.sub(r"\s*\.\s*", ".", java_code)
    
    # Fix multiple spaces to single space
    java_code = re.sub(r"\s+", " ", java_code)
    
    # Ensure proper line breaks after semicolons and braces
    java_code = re.sub(r";\s*", ";\n", java_code)
    java_code = re.sub(r"\{\s*", "{\n", java_code)
    java_code = re.sub(r"\}\s*", "}\n", java_code)
    
    # Ensure proper indentation for better readability
    formatted_lines = []
    indent_level = 0
    for line in java_code.split("\n"):
        stripped = line.strip()
        if stripped.endswith("}"):
            indent_level -= 1

        formatted_lines.append("    " * max(indent_level, 0) + stripped)

        if stripped.endswith("{"):
            indent_level += 1

    return "\n".join(formatted_lines).strip()

In [ ]:
def format_java_code_bk(java_code: str):
    return java_code

### Execute

In [8]:
def extract_rule_label(path):
    match = re.search(r'rule_\w+', path)  # Capture 'rule_' followed by one or more word characters
    return match.group(0) if match else None

In [9]:
def compute_relative_score(actual_value, all_values, epsilon=1e-15):
    relative_value = (actual_value - min(all_values)) / (max(all_values) - min(all_values) + epsilon)
    return relative_value 

In [10]:
def compute_relative_outcome_scores_per_setting(s_file_member, s_file_non_member, outcomes):
    s_file_member = s_file_member.copy()
    
    for outcome in outcomes:
        combined_outcome_values = pd.concat([s_file_member[outcome], s_file_non_member[outcome]], ignore_index=True)
        s_file_member[f'relative_{outcome}'] = s_file_member[outcome].apply(lambda x: compute_relative_score(x, combined_outcome_values))

    return s_file_member


In [11]:
def load_json_files_to_dicts(directory):
    control_dict = {}
    treatment_dict = {}

    def process_dataframe(df, code_column, length_column, group, rule_label):
        df[code_column] = df[code_column].map(format_java_code)
        df = append_covariates(df, code_column, 't')
        df['java_code_infered'] = df['java_code_infered'].map(lambda code: format_java_code(code[0]))
        df = append_covariates(df, 'java_code_infered', 'e')
        
        drop_cols = ['java_code_original', 'length_token_original', 'member'] if group == 'treatment' else ['java_code_transformed', 'length_token_transformed', 'member']
        rename_map = {code_column: 'java_code', length_column: 't_length_token'}

        df = df.drop(columns=drop_cols, errors='ignore').rename(columns=rename_map)
        df['group'] = group
        df['model'] = params['dataset']['model']
        df['rule_number'] = rule_label
        
        return df

    for root, _, files in os.walk(directory):
        if any(item in files for item in ['Null_Hypothesis_member.json', 'Null_Hypothesis_non_member.json', 'Treatment_Hypothesis_member.json', 'Treatment_Hypothesis_non_member.json']):
            try:
                rule_label = extract_rule_label(root)
                print(f"=========================== PROCESSING RULE {rule_label} ==========================")
                try:
                    s1_file_member = pd.read_json(os.path.join(root, 'Null_Hypothesis_member.json'))
                    s1_file_non_member = pd.read_json(os.path.join(root, 'Null_Hypothesis_non_member.json'))
                    s2_file_1_member = pd.read_json(os.path.join(root, 'Treatment_Hypothesis_member.json'))
                    s2_file_2_non_member = pd.read_json(os.path.join(root, 'Treatment_Hypothesis_non_member.json'))
                except Exception as e:
                    print(f"Error loading JSON files in {root}: {e}")

                control_members = compute_relative_outcome_scores_per_setting(s1_file_member, s1_file_non_member, ['loss', 'zlib', 'min_k'])
                treatment_members = compute_relative_outcome_scores_per_setting(s2_file_1_member, s2_file_2_non_member, ['loss', 'zlib', 'min_k'])

                control_dict[rule_label] = process_dataframe(control_members, 'java_code_original', 'length_token_original', 'control', rule_label)
                treatment_dict[rule_label] = process_dataframe(treatment_members, 'java_code_transformed', 'length_token_transformed', 'treatment', rule_label)
            except Exception as e:
                print(f"Error processing directory {root}: {e}")

    return control_dict, treatment_dict


In [ ]:
control_dict, treatment_dict = load_json_files_to_dicts(f"{params['dataset']['path']}/{params['dataset']['model']}")

In [ ]:
combined_control_df = pd.concat(control_dict.values(), ignore_index=True)
combined_control_df.reset_index(drop=True)
treatment_dict_df = pd.concat(treatment_dict.values(), ignore_index=True)
treatment_dict_df.reset_index(drop=True)

### Storing data

In [ ]:
print(f"===================== SAVING DATA FOR {params['dataset']['model']} ==================m ")
output_path = f"{params['output_dir']}/{params['dataset']['model']}"
create_folder(output_path)
combined_control_df.to_json(f"{output_path}/control.json")
treatment_dict_df.to_json(f"{output_path}/treatment.json")